# 傾向スコアを用いた因果推論

Lalondeデータセット（NSW処置群 + CPS比較群）を用いて、傾向スコアによる因果推論を実施する。

## Step 0: 環境構築・データ読み込み

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression

plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

In [ ]:
df_org = pd.read_csv('data/ec675_nsw.tab', sep='\t')
df_org.head()

In [ ]:
df_org.info()

In [ ]:
df_org['sample'].value_counts()

### 分析用データの作成

- NSW処置群（sample=1, treated=1）とCPS比較群（sample=2）を抽出
- CPS群のtreatedを0に設定

In [ ]:
df_nsw_treated = df_org[(df_org['sample'] == 1) & (df_org['treated'] == 1)].copy()
df_cps = df_org[df_org['sample'] == 2].copy()

df_cps['treated'] = 0

df_analysis = pd.concat([df_nsw_treated, df_cps], ignore_index=True)

print(f"処置群: {df_analysis['treated'].sum()}名")
print(f"対照群: {(df_analysis['treated'] == 0).sum()}名")
print(f"合計: {len(df_analysis)}名")

### ベンチマーク効果の算出

NSW RCTデータ（sample=1）の処置群と対照群の差分をATTとして算出

In [ ]:
df_nsw_rct = df_org[df_org['sample'] == 1].copy()

y_treat_rct = df_nsw_rct[df_nsw_rct['treated'] == 1]['re78']
y_control_rct = df_nsw_rct[df_nsw_rct['treated'] == 0]['re78']

att_rct = y_treat_rct.mean() - y_control_rct.mean()

print(f"RCTによるベンチマークATT: ${att_rct:.2f}")

## Step 1: 探索的データ分析（EDA）

### 基本統計量の比較

In [ ]:
covariates = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']

df_treat = df_analysis[df_analysis['treated'] == 1]
df_control = df_analysis[df_analysis['treated'] == 0]

stats_treat = df_treat[covariates].describe().T[['mean', 'std']]
stats_control = df_control[covariates].describe().T[['mean', 'std']]

stats_treat.columns = ['mean_treat', 'std_treat']
stats_control.columns = ['mean_control', 'std_control']

df_stats = pd.concat([stats_treat, stats_control], axis=1)
df_stats

### 標準化平均差（SMD）の算出

$$
SMD = \frac{\bar{X}_{treat} - \bar{X}_{control}}{\sqrt{(s^2_{treat} + s^2_{control})/2}}
$$

目安: |SMD| < 0.1 でバランス良好

In [ ]:
def calculate_smd(df, covariates, treatment_col='treated'):
    """
    Calculate Standardized Mean Difference for covariates
    """
    df_treat = df[df[treatment_col] == 1]
    df_control = df[df[treatment_col] == 0]
    
    smd_dict = {}
    
    for cov in covariates:
        mean_treat = df_treat[cov].mean()
        mean_control = df_control[cov].mean()
        var_treat = df_treat[cov].var()
        var_control = df_control[cov].var()
        
        pooled_std = np.sqrt((var_treat + var_control) / 2)
        smd = (mean_treat - mean_control) / pooled_std
        
        smd_dict[cov] = smd
    
    return pd.Series(smd_dict)

In [ ]:
smd_before = calculate_smd(df_analysis, covariates)

df_smd = pd.DataFrame({
    'SMD_before': smd_before
})

df_smd

調整前の標準化平均差を確認すると、多くの共変量でバランスが取れていないことがわかる。

## Step 2: 傾向スコアの推定

### ロジスティック回帰による傾向スコアの推定

In [ ]:
X = df_analysis[covariates]
y = df_analysis['treated']

model_ps = LogisticRegression(max_iter=1000, random_state=42)
model_ps.fit(X, y)

df_analysis['propensity_score'] = model_ps.predict_proba(X)[:, 1]

df_analysis[['treated', 'propensity_score']].head(10)

In [ ]:
print("傾向スコアの基本統計量:")
df_analysis.groupby('treated')['propensity_score'].describe()

### 傾向スコアの分布の可視化

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bins = np.arange(0, 1.05, 0.05)

ax.hist(df_analysis[df_analysis['treated'] == 1]['propensity_score'], 
        bins=bins, alpha=0.5, label='処置群', color='blue', edgecolor='black')
ax.hist(df_analysis[df_analysis['treated'] == 0]['propensity_score'], 
        bins=bins, alpha=0.5, label='対照群', color='red', edgecolor='black')

ax.set_xlabel('傾向スコア')
ax.set_ylabel('サンプル数')
ax.set_title('傾向スコアの分布')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 共通サポート仮定の確認

各binで処置群・対照群の両方にサンプルが存在するかを確認

In [ ]:
bins = np.arange(0, 1.05, 0.05)

df_analysis['ps_bin'] = pd.cut(df_analysis['propensity_score'], bins=bins)

overlap_check = df_analysis.groupby(['ps_bin', 'treated']).size().unstack(fill_value=0)
overlap_check['both_present'] = (overlap_check[0] > 0) & (overlap_check[1] > 0)

print("共通サポート仮定の確認:")
print(f"両群が存在するbin: {overlap_check['both_present'].sum()} / {len(overlap_check)}")
print("\n各binのサンプル数:")
overlap_check